In [1]:
"""Run two below lines to get my_abc module"""
# !git clone https://github.com/thanhttttt/thanh.git
# !pip install -r /content/thanh/requirements.txt
"""Run two below lines to drive"""
# from google.colab import drive
# drive.mount('/content/drive')

'Run two below lines to drive'

# Import

In [2]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


import sys
sys.path.append('../')
sys.path.append('/content/thanh/')
sys.path.append('../thanh/')

import sionna

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible random number generation

# Load the required Sionna components
from sionna.nr.my_abc import *

# Load model weight

In [3]:
_model = CustomNeuralReceiver(training = False)
inputs = tf.zeros([1,48,14,18])
_model(inputs)
_model.summary()

#load_weights(_model, '/content/drive/MyDrive/Pusch_data/Model_weights/model_weight_FULL_RB_epoch_40.pkl')
# load_weights(_model, '../model_weight_FULL_RB_epoch_40.pkl')
load_weights(_model, '../weight_4RB_batchsize_1024_186k_sample_dynamic_config_epoch120.pkl')

Model: "custom_neural_receiver"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             multiple                  20864     
                                                                 
 residual_block (ResidualBl  multiple                  639232    
 ock)                                                            
                                                                 
 residual_block_1 (Residual  multiple                  639232    
 Block)                                                          
                                                                 
 residual_block_2 (Residual  multiple                  639232    
 Block)                                                          
                                                                 
 residual_block_3 (Residual  multiple                  639232    
 Block)                                     

# Set up config

In [4]:
"""test setup"""
batch_size = 1

"""nrb config setup"""
RB_start = 0
NRB = 162
PCI = 443
RNTI = 40035
MCS = 8

"""channel setup"""
no = 2.
CDL_model = 'A'
delay_spread = 50
speed = 1

## create channel

In [5]:
channel_model = CDL(model = CDL_model,
                            delay_spread = delay_spread*1e-9,
                            carrier_frequency = CARRIER_FREQUENCY,
                            ut_array = Ue_Antenna,
                            bs_array = Gnb_AntennaArray,
                            direction = 'uplink',
                            min_speed = speed,
                            max_speed = speed)

## create 4RB samples

In [6]:
"""default config is 4RB"""
"""Pusch config"""
"""NRB (Number of Resource Blocks) and MCS (Modulation and Coding Scheme) are fixed across all samples."""
NRB = 4
MCS = 9
"""Prototype configuration generation for system and UE settings"""
sysCfg = SystemConfig(
                    NCellId = 442,
                    FrequencyRange = 1,
                    BandWidth = 60,
                    Numerology = 1,
                    CpType = 0,
                    BwpNRb = 162,
                    BwpRbOffset = 0,
                    harqProcFlag = 0,
                    nHarqProc = 1,
                    rvSeq = 0
                )
ueCfg = UeConfig(
                TransformPrecoding = 0,
                Rnti = 40035,
                nId = 442,
                CodeBookBased = 0,
                DmrsPortSetIdx = [0],
                NLayers = 1,
                NumDmrsCdmGroupsWithoutData = 2,
                Tpmi = 0,
                FirstSymb = 0,
                NPuschSymbAll = 14,
                RaType = 1,
                FirstPrb = 0,
                NPrb = NRB,
                FrequencyHoppingMode = 0,
                McsTable = 0,
                Mcs = MCS,
                ILbrm = 0,
                nScId = 0,
                NnScIdId = 442,
                DmrsConfigurationType = 0,
                DmrsDuration = 1,
                DmrsAdditionalPosition = 1,
                PuschMappingType = 0,
                DmrsTypeAPosition = 3,
                HoppingMode = 0,
                NRsId = 0,
                Ptrs = 0,
                ScalingFactor = 0,
                OAck = 0,
                IHarqAckOffset = 11,
                OCsi1 = 0,
                ICsi1Offset = 7,
                OCsi2 = 0,
                ICsi2Offset = 0,
                NPrbOh = 0,
                nCw = 1,
                TpPi2Bpsk = 0
            )
myCfg = MyConfig(sysCfg, [ueCfg])
puschCfg = MyPUSCHConfig(myCfg)
# puschCfg.show() # uncomment for detail
simulator = MySimulator(puschCfg)
channel = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid,
                                    add_awgn=False, normalize_channel=True, return_channel=True)
b1, c1, y1, x1 ,h1 = simulator.sim(batch_size, channel, no, return_tx_iq=True, return_channel=True, gen_prng_seq=20044)
r1 = simulator.ref(batch_size)
np.sum(b1), np.sum(c1), np.sum(x1), np.sum(r1)

hello (1, 1, 1152) (1, 1, 1152) False


(374.0, 583.0, (13.899494-7.79899j), (4+12j))

In [7]:
([1234],)(0)

<>:1: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
<>:1: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_50000/3595707216.py:1: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ([1234],)(0)
/tmp/ipykernel_50000/3595707216.py:1: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ([1234],)(0)
/tmp/ipykernel_50000/3595707216.py:1: SyntaxWarning: 'tuple' object is not callable; perhaps you missed a comma?
  ([1234],)(0)


TypeError: 'tuple' object is not callable

In [ ]:
simulator.TB_Encoder.scrambler.__dict__

{'_self_setattr_tracking': True,
 '_obj_reference_counts_dict': ObjectIdentityDictionary({<_ObjectIdentityWrapper wrapping False>: 1, <_ObjectIdentityWrapper wrapping True>: 3, <_ObjectIdentityWrapper wrapping ListWrapper([1311867322])>: 1, <_ObjectIdentityWrapper wrapping TensorShape([1, 1, 1152])>: 1, <_ObjectIdentityWrapper wrapping <tf.Tensor 'tb_encoder_5/tb5g_scrambler_5/Reshape:0' shape=(1, 1, 1152) dtype=float32>>: 1}),
 '_auto_get_config': False,
 '_instrumented_keras_api': True,
 '_instrumented_keras_layer_class': True,
 '_instrumented_keras_model_class': False,
 '_trainable': True,
 '_stateful': False,
 'built': True,
 '_input_spec': None,
 '_build_input_shape': TensorShape([1, 1, 1152]),
 '_saved_model_inputs_spec': TensorSpec(shape=(1, 1, 1152), dtype=tf.float32, name=None),
 '_saved_model_arg_spec': ([TensorSpec(shape=(1, 1, 1152), dtype=tf.float32, name=None)],
  {}),
 '_supports_masking': False,
 '_name': 'tb5g_scrambler_5',
 '_activity_regularizer': None,
 '_trainable_

In [29]:
"""default config is 4RB"""
"""Pusch config"""
"""NRB (Number of Resource Blocks) and MCS (Modulation and Coding Scheme) are fixed across all samples."""
NRB = 4
MCS = 9
"""Prototype configuration generation for system and UE settings"""
sysCfg = SystemConfig(
                    NCellId = 441,
                    FrequencyRange = 1,
                    BandWidth = 60,
                    Numerology = 1,
                    CpType = 0,
                    BwpNRb = 162,
                    BwpRbOffset = 0,
                    harqProcFlag = 0,
                    nHarqProc = 1,
                    rvSeq = 0
                )
ueCfg = UeConfig(
                TransformPrecoding = 0,
                Rnti = 40035,
                nId = 441,
                CodeBookBased = 0,
                DmrsPortSetIdx = [0],
                NLayers = 1,
                NumDmrsCdmGroupsWithoutData = 2,
                Tpmi = 0,
                FirstSymb = 0,
                NPuschSymbAll = 14,
                RaType = 1,
                FirstPrb = 0,
                NPrb = NRB,
                FrequencyHoppingMode = 0,
                McsTable = 0,
                Mcs = MCS,
                ILbrm = 0,
                nScId = 0,
                NnScIdId = 441,
                DmrsConfigurationType = 0,
                DmrsDuration = 1,
                DmrsAdditionalPosition = 1,
                PuschMappingType = 0,
                DmrsTypeAPosition = 3,
                HoppingMode = 0,
                NRsId = 0,
                Ptrs = 0,
                ScalingFactor = 0,
                OAck = 0,
                IHarqAckOffset = 11,
                OCsi1 = 0,
                ICsi1Offset = 7,
                OCsi2 = 0,
                ICsi2Offset = 0,
                NPrbOh = 0,
                nCw = 1,
                TpPi2Bpsk = 0
            )
myCfg = MyConfig(sysCfg, [ueCfg])
puschCfg = MyPUSCHConfig(myCfg)
simulator = MySimulator(puschCfg)
channel = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid,
                                    add_awgn=False, normalize_channel=True, return_channel=True)
# puschCfg.show() # uncomment for detail
b, c, y, x ,h = simulator.sim(batch_size, channel, no, return_tx_iq=True, return_channel=True, gen_prng_seq=20044)
r = simulator.ref(batch_size)
np.sum(b), np.sum(c), np.sum(x), np.sum(r)

hello (1, 1, 1152) (1, 1, 1152) False


(374.0, 566.0, (15.798988-5.656856j), (-4+0j))

In [30]:
from sionna.utils import expand_to_rank

In [21]:
c_init = 40035 * 2**15 + 442

In [25]:
simulator.update_c_init([c_init])
simulator.TB_Encoder.scrambler.build([1,1,1152])
b, c, y, x ,h = simulator.sim(batch_size, channel, no, return_tx_iq=True, return_channel=True, gen_prng_seq=20044)
r = simulator.ref(batch_size)
print(np.sum(b), np.sum(c), np.sum(x), np.sum(r))

374.0 566.0 (15.798988-5.656856j) (-4+0j)


In [26]:
simulator.TB_Encoder.scrambler.__dict__

{'_self_setattr_tracking': True,
 '_obj_reference_counts_dict': ObjectIdentityDictionary({<_ObjectIdentityWrapper wrapping False>: 1, <_ObjectIdentityWrapper wrapping True>: 3, <_ObjectIdentityWrapper wrapping ListWrapper([1311867322])>: 1, <_ObjectIdentityWrapper wrapping ListWrapper([1, 1, 1152])>: 1, <_ObjectIdentityWrapper wrapping <tf.Tensor: shape=(1, 1, 1152), dtype=float32, numpy=array([[[1., 0., 1., ..., 1., 0., 1.]]], dtype=float32)>>: 1}),
 '_auto_get_config': False,
 '_instrumented_keras_api': True,
 '_instrumented_keras_layer_class': True,
 '_instrumented_keras_model_class': False,
 '_trainable': True,
 '_stateful': False,
 'built': True,
 '_input_spec': None,
 '_build_input_shape': TensorShape([1, 1, 1152]),
 '_saved_model_inputs_spec': TensorSpec(shape=(1, 1, 1152), dtype=tf.float32, name=None),
 '_saved_model_arg_spec': ([TensorSpec(shape=(1, 1, 1152), dtype=tf.float32, name=None)],
  {}),
 '_supports_masking': False,
 '_name': 'tb5g_scrambler_4',
 '_activity_regularize

In [16]:
"""default config is 4RB"""
"""Pusch config"""
"""NRB (Number of Resource Blocks) and MCS (Modulation and Coding Scheme) are fixed across all samples."""
NRB = 4
MCS = 9
"""Prototype configuration generation for system and UE settings"""
sysCfg = SystemConfig(
                    NCellId = 442,
                    FrequencyRange = 1,
                    BandWidth = 60,
                    Numerology = 1,
                    CpType = 0,
                    BwpNRb = 162,
                    BwpRbOffset = 0,
                    harqProcFlag = 0,
                    nHarqProc = 1,
                    rvSeq = 0
                )
ueCfg = UeConfig(
                TransformPrecoding = 0,
                Rnti = 40035,
                nId = 442,
                CodeBookBased = 0,
                DmrsPortSetIdx = [0],
                NLayers = 1,
                NumDmrsCdmGroupsWithoutData = 2,
                Tpmi = 0,
                FirstSymb = 0,
                NPuschSymbAll = 14,
                RaType = 1,
                FirstPrb = 0,
                NPrb = NRB,
                FrequencyHoppingMode = 0,
                McsTable = 0,
                Mcs = MCS,
                ILbrm = 0,
                nScId = 0,
                NnScIdId = 442,
                DmrsConfigurationType = 0,
                DmrsDuration = 1,
                DmrsAdditionalPosition = 1,
                PuschMappingType = 0,
                DmrsTypeAPosition = 3,
                HoppingMode = 0,
                NRsId = 0,
                Ptrs = 0,
                ScalingFactor = 0,
                OAck = 0,
                IHarqAckOffset = 11,
                OCsi1 = 0,
                ICsi1Offset = 7,
                OCsi2 = 0,
                ICsi2Offset = 0,
                NPrbOh = 0,
                nCw = 1,
                TpPi2Bpsk = 0
            )
myCfg = MyConfig(sysCfg, [ueCfg])
puschCfg = MyPUSCHConfig(myCfg, 5)
simulator2 = MySimulator(puschCfg)
channel = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid,
                                    add_awgn=False, normalize_channel=True, return_channel=True)
# puschCfg.show() # uncomment for detail
b2, c2, y2, x2 ,h2 = simulator2.sim(batch_size, channel, no, return_tx_iq=True, return_channel=True, gen_prng_seq=20044)
r2 = simulator2.ref(batch_size)
np.sum(b2), np.sum(c2), np.sum(x2), np.sum(r2)

hello (1, 1, 1152) (1, 1, 1152) False


(374.0, 583.0, (3.8994942-19.798988j), (-6+0j))

In [31]:
simulator.Resource_Grid_Mapper._resource_grid.pilot_pattern.pilots

<tf.Variable 'Variable:0' shape=(1, 1, 96) dtype=complex64, numpy=
array([[[ 1.+1.j,  0.+0.j,  1.-1.j,  0.+0.j, -1.-1.j,  0.+0.j, -1.+1.j,
          0.+0.j, -1.+1.j,  0.+0.j,  1.+1.j,  0.+0.j, -1.+1.j,  0.+0.j,
         -1.-1.j,  0.+0.j, -1.+1.j,  0.+0.j, -1.+1.j,  0.+0.j,  1.-1.j,
          0.+0.j,  1.+1.j,  0.+0.j, -1.+1.j,  0.+0.j,  1.-1.j,  0.+0.j,
         -1.-1.j,  0.+0.j,  1.-1.j,  0.+0.j,  1.-1.j,  0.+0.j,  1.+1.j,
          0.+0.j,  1.+1.j,  0.+0.j, -1.-1.j,  0.+0.j,  1.-1.j,  0.+0.j,
         -1.+1.j,  0.+0.j,  1.-1.j,  0.+0.j, -1.+1.j,  0.+0.j,  1.+1.j,
          0.+0.j,  1.-1.j,  0.+0.j,  1.+1.j,  0.+0.j,  1.-1.j,  0.+0.j,
         -1.-1.j,  0.+0.j,  1.-1.j,  0.+0.j,  1.-1.j,  0.+0.j, -1.-1.j,
          0.+0.j, -1.-1.j,  0.+0.j,  1.+1.j,  0.+0.j, -1.-1.j,  0.+0.j,
          1.+1.j,  0.+0.j,  1.+1.j,  0.+0.j, -1.-1.j,  0.+0.j, -1.+1.j,
          0.+0.j, -1.-1.j,  0.+0.j, -1.-1.j,  0.+0.j, -1.+1.j,  0.+0.j,
         -1.-1.j,  0.+0.j, -1.-1.j,  0.+0.j, -1.+1.j,  0.+0.j,  1.+1.

## create N-RB samples

In [9]:
sysCfg = SystemConfig(
                    NCellId = PCI,
                    FrequencyRange = 1,
                    BandWidth = 60,
                    Numerology = 1,
                    CpType = 0,
                    NTxAnt = 1,
                    NRxAnt = 8,
                    BwpNRb = 162,
                    BwpRbOffset = 0,
                    harqProcFlag = 0,
                    nHarqProc = 1,
                    rvSeq = 0
                )
ueCfg = UeConfig(
                TransformPrecoding = 0,
                Rnti = RNTI,
                nId = PCI,
                CodeBookBased = 0,
                DmrsPortSetIdx = [0],
                NLayers = 1,
                NumDmrsCdmGroupsWithoutData = 2,
                Tpmi = 0,
                FirstSymb = 0,
                NPuschSymbAll = 14,
                RaType = 1,
                FirstPrb = RB_start,
                NPrb = NRB,
                FrequencyHoppingMode = 0,
                McsTable = 0,
                Mcs = MCS,
                ILbrm = 0,
                nScId = 0,
                NnScIdId = PCI,
                DmrsConfigurationType = 0,
                DmrsDuration = 1,
                DmrsAdditionalPosition = 1,
                PuschMappingType = 0,
                DmrsTypeAPosition = 3,
                HoppingMode = 0,
                NRsId = 0,
                Ptrs = 0,
                ScalingFactor = 0,
                OAck = 0,
                IHarqAckOffset = 11,
                OCsi1 = 0,
                ICsi1Offset = 7,
                OCsi2 = 0,
                ICsi2Offset = 0,
                NPrbOh = 0,
                nCw = 1,
                TpPi2Bpsk = 0
            )
myCfg = MyConfig(sysCfg, [ueCfg])
puschCfg = MyPUSCHConfig(myCfg)
# puschCfg.show() # uncomment for detail

In [10]:
simulator = MySimulator(puschCfg)
channel = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid,
                                    add_awgn=False, normalize_channel=True, return_channel=True)

In [11]:
b, c, y, x ,h = simulator.sim(batch_size, channel, no, return_tx_iq=True, return_channel=True)
r = simulator.ref(batch_size)

# Evaluate

In [13]:
"""Evaluate"""
preds = predict(_model, y, r)
c_pred = tf.reshape(preds, [preds.shape[0], 1, c.shape[-1]])
b_hat, crc = simulator.TB_Decoder(c_pred)
# loss_cal(c_pred, c)
compute_ber(b, b_hat), crc

(<tf.Tensor: shape=(), dtype=float64, numpy=0.0>,
 <tf.Tensor: shape=(8, 1), dtype=bool, numpy=
 array([[ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True],
        [ True]])>)

In [14]:
h_est, x_hat, llr_det, b_hat, crc = simulator.rec(y)
compute_ber(b, b_hat), crc

(<tf.Tensor: shape=(), dtype=float64, numpy=0.1395899623951403>,
 <tf.Tensor: shape=(8, 1), dtype=bool, numpy=
 array([[False],
        [False],
        [False],
        [False],
        [False],
        [False],
        [False],
        [False]])>)